# Train random forest classifiers for variant filtering

In previous work (all done in R) I showed that a RF, using most of the features produced by rastair2, can distinguish SNPs from REF positions in 15x sequenced TAPS data with ~99.7% accuracy, and approx 90% PPV.

I would like to perform this kind of filtering in the actual rastair code, but to do so, I need to create a serialisable version of the classifiers.

It seems to be impossible to train in R and export into a format that Rust can understand, so instead I will just write some code here to perform the training in Rust directly.

To achieve this, we need to:

1. Run rastair on data with labels and extract all the features from the vcf
2. Transform the variant info data into a matrix that can be used for training. This requires:
    1. One-hot encoding ref/alt alleles and sequence context
    2. Normalise count-based values to depth
    3. Calculate any derived scores
    4. For CpGs or putative CpGs, include some of the features from position before/after in the training
3. Train with 10-fold cross-validation, ntree=1000 and mtry=2
4. Serialise the resulting object

## 0. Install dependencies

In [3]:
:dep ndarray = { version = "^0.16.1", default-features = true }
:dep flate2 = { version = "^1.1.2" }
:dep chrono = { version = "^0.4.41" }
:dep serde = { version = "^1.0.219" }
:dep bincode = { version = "2.0.1", features = ["serde"] }
:dep rand = { version = "^0.9.1" }
:dep anyhow = { version = "1.0.98" }
:dep smartcore = { version = "^0.4.1", features = ["serde", "ndarray-bindings"] }
//:dep rust-htslib = { version = "^0.49.0", git = "https://github.com/killercup/rust-htslib.git", branch = "feature/cstr8", default-features = false, features = ["bzip2", "lzma", "libdeflate"] }
:dep csv = { version = "^1.3.1" }

## 1. Load training data
We will produce the training data in R since that code is already there and ready to go. See `vcf_to_train.ipynb` for details.

In [4]:
use anyhow::{bail, Result};
use std::{fs::File, error::Error, io::{BufRead, BufReader, Read}};
use csv::{Reader, ReaderBuilder};
use ndarray::prelude::*;
use ndarray::{Axis, concatenate};
use flate2::read::GzDecoder;

enum FeatureType {
    CpG,
    NewCpG,
    Other
}
use FeatureType::*;

fn _read_all_rows<R: std::io::Read>(reader: &mut Reader<R>) -> Result<(Vec<f32>, Vec<u32>)> {
    // Build the CSV reader and iterate over each record.
    let mut output_vector: Vec<f32> = Vec::with_capacity(2_000_000*57);
    let mut labels: Vec<u32> = Vec::with_capacity(2_000_000);
    let mut row: usize = 0;
    for result in reader.records() {
        row = row+1;
        // The iterator yields Result<StringRecord, Error>, so we check the
        // error here..
        let record = result?;
        let num_columns = record.len();

        let label = record.get(num_columns-1).unwrap_or("REF");
        if label == "REF" {
            labels.push(0);
        } else {
            labels.push(1);
        }

        for i in 1..(num_columns-2) {
            let v = record.get(i).unwrap_or("0.0");
            output_vector.push(v.parse().unwrap_or_default());
        }
    }
    Ok((output_vector, labels))
}

fn tsv_to_matrix(path: &str, ftype: FeatureType) -> Result<(ndarray::Array2<f32>, ndarray::Array1<u32>)> {
    let file = File::open(path)?;
    let decoder = GzDecoder::new(file);
    let reader = BufReader::new(decoder);

    let mut rdr = ReaderBuilder::new().delimiter(b'\t').from_reader(reader);
    let (data, labels) = _read_all_rows(&mut rdr)?;
    let n_observations = labels.len();
    let num_features: usize = match ftype {
        CpG => 56,
        NewCpG => 56,
        Other   => 54
    };
    println!("Finished reading {} rows into a vector of size {}", n_observations, data.len());

    let data_matrix = ndarray::Array::from_shape_vec((n_observations, num_features), data)?;
    Ok((data_matrix, ndarray::Array::from_vec(labels)))
}

let cpg_data_file = "data/chr12_CpG_features.txt.gz";
let (full_data_matrix_cpg, labels) = tsv_to_matrix(cpg_data_file, CpG).unwrap();


Finished reading 2206920 rows into a vector of size 123587520


Done reading training data. Now we need to subsample a set of rows and create a slice of the original matrix with only those rows for training.

In [5]:
use rand::prelude::*;
use rand::seq::SliceRandom;

fn select_random_elements(
    array: &Array1<u32>,
    num_samples: usize,
    to_select: u32
) -> Result<Array1<usize>, String> {
    // Find indices of positive elements
    let valid_indices: Vec<usize> = array
        .iter()
        .enumerate()
        .filter(|x| *x.1 == to_select)
        .map(|(idx, _)| idx)
        .collect();

    if valid_indices.len() < num_samples {
        return Err(format!(
            "Not enough positive elements. Found {} positive values, but need {}",
            valid_indices.len(),
            num_samples
        ));
    }

    // Randomly sample indices
    let mut rng = rand::rng();

    let selected_indices: Vec<usize> = valid_indices
        .choose_multiple(&mut rng, num_samples)
        .cloned()
        .collect();

    Ok(Array::from_vec(selected_indices))
}

fn select_rows_by_indices(array: &Array2<f32>, indices: &Array1<usize>) -> Result<Array2<f32>> {
    // Method 1: Using ndarray's select method (most efficient)
    let selected = array.select(Axis(0), indices.as_slice().unwrap());
    Ok(selected)
}

fn combine_and_sort(arr1: Array1<usize>, arr2: Array1<usize>) -> Array1<usize> {
    // Method 1: Using ndarray's concatenate function
    let combined = concatenate![Axis(0), arr1.view(), arr2.view()];

    // Convert to Vec, sort, and convert back
    let mut values: Vec<usize> = combined.to_vec();
    values.sort_by(|a, b| a.partial_cmp(b).unwrap());

    Array1::from_vec(values)
}

let true_pos_subset = select_random_elements(&labels, 4000, 1).unwrap();
let true_neg_subset = select_random_elements(&labels, 16000, 0).unwrap();

let all_indices = combine_and_sort(true_pos_subset, true_neg_subset);

let training_matrix = select_rows_by_indices(&full_data_matrix_cpg, &all_indices).unwrap();
let label_subset = labels.select(Axis(0), all_indices.as_slice().unwrap());

let training_dim=training_matrix.dim();
println!("Will train from matrix with dimensions {}x{}", training_dim.0, training_dim.1);

Will train from matrix with dimensions 20000x56


We should now have everything we need to train our RF. We will perform 10-fold CV to find the best parameter set:

In [ ]:
//use smartcore::linalg::basic::arrays::{Array2};
use smartcore::ensemble::random_forest_classifier::{RandomForestClassifier, RandomForestClassifierParameters};
use smartcore::tree::decision_tree_classifier::SplitCriterion;
use smartcore::linalg::{ndarray::*, basic::matrix::DenseMatrix};
use smartcore::metrics::precision;
use smartcore::api::SupervisedEstimator;
use smartcore::model_selection::{train_test_split, cross_validate, KFold};
use chrono::Utc;
// This is needed because there's a bug in smartcore that means that trait bounds are incompatible between RandomForest::fit and precision
fn my_precision(y_true: &Vec<u32>, y_pred: &Vec<u32>) -> f64 {
    let y_true_f32: Vec<f32> = y_true.iter().map(|x| *x as f32).collect();
    let y_pred_f32: Vec<f32> = y_pred.iter().map(|x| *x as f32).collect();
    precision(&y_true_f32, &y_pred_f32)
}

fn grid_search(x_train: &DenseMatrix<f32>, y_train: &Vec<u32>) -> Option<(u16, usize, usize)> {
    let mut best_score = 0.;
    let mut best_params = None;

    let n_trees: Vec<u16> = vec![500, 1000, 1500];
    let min_samples_split: Vec<usize> = vec![2, 3, 4];
    let max_features: Vec<usize> = vec![2, 3, 4, 7];

    for n_tree in &n_trees {
        for m_split in &min_samples_split {
            for m_feat in &max_features {
                eprintln!("{} - Training with parameters n_tree: {} min_ss: {} mtry: {}", Utc::now().to_rfc3339(), n_tree, m_split, m_feat);

                let cv_score = cross_validate(
                    RandomForestClassifier::new(),
                    x_train,
                    y_train,
                    RandomForestClassifierParameters::default()
                        .with_criterion(SplitCriterion::Gini)
                        .with_n_trees(*n_tree)
                        .with_m(*m_feat)
                        .with_min_samples_split(*m_split),
                &KFold::default().with_n_splits(5),
                &my_precision
                ).unwrap();

                if cv_score.mean_test_score() > best_score {
                    best_score = cv_score.mean_test_score();
                    best_params = Some((n_tree, m_feat, m_split));
                }
            }
        }
    }

    println!("Best score: {}", best_score);
    println!("Best params: {:?}", best_params);

    Some(
        (
            *best_params.unwrap().0,
            *best_params.unwrap().1,
            *best_params.unwrap().2
        )
    )
}
#[allow(deprecated)]
let dm = DenseMatrix::new(training_dim.0, training_dim.1, training_matrix.into_raw_vec(), false).unwrap();
let best_params = grid_search(&dm, &label_subset.to_vec()).unwrap_or((1000, 2, 2));

2025-07-13T08:28:13.366720+00:00 - Training with parameters n_tree: 500 min_ss: 2 mtry: 2
2025-07-13T08:29:21.430642+00:00 - Training with parameters n_tree: 500 min_ss: 2 mtry: 3
2025-07-13T08:30:43.463170+00:00 - Training with parameters n_tree: 500 min_ss: 2 mtry: 4
2025-07-13T08:32:41.095116+00:00 - Training with parameters n_tree: 500 min_ss: 2 mtry: 7
2025-07-13T08:35:37.527258+00:00 - Training with parameters n_tree: 500 min_ss: 3 mtry: 2
2025-07-13T08:43:10.399495+00:00 - Training with parameters n_tree: 500 min_ss: 3 mtry: 3
2025-07-13T08:46:34.172049+00:00 - Training with parameters n_tree: 500 min_ss: 3 mtry: 4
2025-07-13T08:51:07.705647+00:00 - Training with parameters n_tree: 500 min_ss: 3 mtry: 7
2025-07-13T09:05:56.134255+00:00 - Training with parameters n_tree: 500 min_ss: 4 mtry: 2
2025-07-13T09:07:05.048780+00:00 - Training with parameters n_tree: 500 min_ss: 4 mtry: 3
2025-07-13T09:08:27.277079+00:00 - Training with parameters n_tree: 500 min_ss: 4 mtry: 4
2025-07-13

Best score: 0.9667013541225747


Best params: Some((500, 7, 4))


In [11]:
use std::fs::File;
use std::io::prelude::*;
use bincode::serde::*;

let mut parameters = RandomForestClassifierParameters::default();
parameters.with_n_trees(best_params.0);
parameters.with_m(best_params.1);
parameters.with_min_samples_split(best_params.2);
parameters.with_criterion(SplitCriterion::Gini);

#[allow(deprecated)]
let classifier = RandomForestClassifier::fit(&dm, &label_subset.to_vec(), parameters).unwrap();

let file_name = "models/SC_RF_CpG.model";
// Save the model
{
    let rf_cpg_bytes = bincode::serde::encode_to_vec(&classifier, bincode::config::standard()).expect("Can not serialize the model");

    File::create(file_name)
        .and_then(|mut f| f.write_all(&rf_cpg_bytes))
        .expect("Can not persist model");
}

Error: unused variable: `rng`

Error: use of moved value: `parameters`

Error: use of moved value: `parameters`

Error: use of moved value: `parameters`

Error: use of moved value: `parameters`